# 43 · DB2 partition writes, batches of 200 and full-row rejects

**Scenario:** Implement a classic-compute writer with batch rollback, per-row fallback and explicit outcome capture.

**Source:** Databricks-provided `samples.nyctaxi.trips` only. File formats, nested structures, replays and histories in this lab are derived from that asset. No hand-created record dataset, third-party download, pandas or generated replacement data is used.

**Prerequisites:** Run notebook 00 first. Each notebook is independent; run its cells in order. Mode: **classic-integration**. Dedicated classic compute + JayDeBeApi + JVM/JDBC jars on every executor + configured secrets/network. External writes off by default.

**Learning-size sample:** `max_rows` defaults to 2,000. Sorting/limiting caps result size, not the underlying scan cost. Assertions use relationships instead of hardcoded dataset counts because sample assets may change.


## Step 1 — Load and inspect the Databricks asset

Read the built-in sample and normalize column names. A stable content fingerprint helps with replay examples; it is not a guaranteed real-world trip key. The original `samples` asset is always read-only.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
dbutils.widgets.text('max_rows','2000')
MAX_ROWS=int(dbutils.widgets.get('max_rows'))
assert 100<=MAX_ROWS<=20000, 'Use 100–20000 rows for these bounded learning labs.'
SOURCE_ASSET='samples.nyctaxi.trips'
try:
    asset=(spark.read.table(SOURCE_ASSET)
        .select('tpep_pickup_datetime','tpep_dropoff_datetime','trip_distance',
                'fare_amount','pickup_zip','dropoff_zip')
        .orderBy('tpep_pickup_datetime','tpep_dropoff_datetime','pickup_zip',
                 'dropoff_zip','trip_distance','fare_amount')
        .limit(MAX_ROWS))
    assert asset.limit(1).count()==1, 'Databricks sample asset is empty.'
except Exception as exc:
    raise RuntimeError('This course requires the Databricks-provided samples.nyctaxi.trips asset. '
        'Check Unity Catalog/sample access on the attached compute. No synthetic or external fallback is used.') from exc
df=(asset.select(F.col('tpep_pickup_datetime').cast('timestamp').alias('pickup_ts'),
    F.col('tpep_dropoff_datetime').cast('timestamp').alias('dropoff_ts'),
    F.col('trip_distance').cast('double').alias('distance'),
    F.col('fare_amount').cast('double').alias('fare'),
    F.col('pickup_zip').cast('string').alias('pickup_zip'),
    F.col('dropoff_zip').cast('string').alias('dropoff_zip')))
df=df.withColumn('trip_id',F.sha2(F.to_json(F.struct(*[F.col(c) for c in df.columns]),
    options={'ignoreNullFields':'false'}),256))
df=df.withColumn('pickup_date',F.to_date('pickup_ts'))
df=df.withColumn('duration_sec',F.unix_timestamp('dropoff_ts')-F.unix_timestamp('pickup_ts'))
print('Source:',SOURCE_ASSET,'sample limit:',MAX_ROWS)
# trip_id is a content fingerprint, NOT an official source/business trip identifier.
# Identical source rows intentionally have identical fingerprints.
df.show(5,truncate=False)


## Workspace setup — Choose course-owned output names

Set `target_catalog` to a catalog where you can create a schema. This lab creates a per-user course schema and uses lab-specific names. Rerunning a write lab can replace its own course tables. Nothing is written to the `samples` catalog.


In [ ]:
LAB='43'

import hashlib, re
dbutils.widgets.text('target_catalog','')
target_catalog=dbutils.widgets.get('target_catalog').strip() or spark.sql('SELECT current_catalog()').first()[0]
assert target_catalog!='samples', 'Set target_catalog to a writable catalog, not the read-only samples catalog.'
user=spark.sql('SELECT current_user()').first()[0]
schema='pyspark_lab_'+hashlib.sha256(user.encode()).hexdigest()[:10]
def quote_identifier(value): return '`'+value.replace('`','``')+'`'
namespace=f'{quote_identifier(target_catalog)}.{quote_identifier(schema)}'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {namespace}')
def table(name):
    assert re.fullmatch(r'[a-z][a-z0-9_]*',name)
    return f'{namespace}.`lab_{LAB}_{name}`'
print('Course-only namespace:',namespace)


## Step 2 — Prepare only real asset rows

This advanced optional lab uses JayDeBeApi, a DB2 JDBC jar available on every executor, and dedicated classic compute (RDD APIs). It uses no pandas. Pre-create an empty dedicated DB2 course table with TRIP_ID as a unique key and compatible columns.


In [ ]:
RUN_DB2=False
DB2_DRIVER='com.ibm.db2.jcc.DB2Driver'
DB2_URL=''; DB2_JAR=''; DB2_TABLE=''
SECRET_SCOPE=''; USER_KEY=''; PASSWORD_KEY=''
BATCH_SIZE=200
NUM_PARTITIONS=2
export=df.select('trip_id','pickup_zip',F.col('fare').cast('decimal(14,2)').alias('fare'),'pickup_ts').dropDuplicates(['trip_id']).limit(400)
print('DB2 enabled:',RUN_DB2,'max export rows:',400,'batch size:',BATCH_SIZE)


**Expected check:** Default mode does not resolve secrets, connect or start any executor-side write.


## Step 3 — Define transactional batch fallback

Each successful batch is committed. If executemany fails, roll back that batch before trying rows separately. NULL stays None. Decimal and datetime values are bound using JDBC BigDecimal/Timestamp objects rather than treating typed values as text. Commit/connection failures can leave outcomes uncertain, so they are reported as failed/unknown, never silently counted as inserted.


In [ ]:
def partition_writer(iterator):
    import jaydebeapi
    import jpype
    from decimal import Decimal
    from datetime import datetime, date
    conn=None
    def outcomes(rows,status,reason=None):
        return [(r['trip_id'],r['pickup_zip'],r['fare'],r['pickup_ts'],status,reason) for r in rows]
    pending=[]
    try:
        conn=jaydebeapi.connect(DB2_DRIVER,DB2_URL,[db2_user,db2_password],DB2_JAR)
        conn.jconn.setAutoCommit(False)
        cursor=conn.cursor()
        def convert_value(value):
            if value is None: return None
            if isinstance(value,Decimal):
                return jpype.JClass('java.math.BigDecimal')(str(value))
            if isinstance(value,datetime):
                if value.tzinfo is not None:
                    raise ValueError('Choose explicit timezone conversion before binding a DB2 timestamp without timezone.')
                return jpype.JClass('java.sql.Timestamp').valueOf(value.strftime('%Y-%m-%d %H:%M:%S.%f'))
            if isinstance(value,date):
                return jpype.JClass('java.sql.Date').valueOf(value.isoformat())
            return value
        def flush(rows):
            values=[tuple(convert_value(r[c]) for c in ['trip_id','pickup_zip','fare','pickup_ts']) for r in rows]
            try:
                cursor.executemany(insert_sql,values)
            except Exception:
                conn.rollback()
                result=[]
                for row,value in zip(rows,values):
                    # Check after rollback; this also makes committed-row task replay a skip.
                    try:
                        cursor.execute(exists_sql,[row['trip_id']])
                        if cursor.fetchone()[0]>0:
                            conn.rollback(); result.extend(outcomes([row],'skipped')); continue
                        cursor.execute(insert_sql,value)
                    except Exception as exc:
                        conn.rollback()
                        # Error class only: database exception text can contain source values or connection details.
                        result.extend(outcomes([row],'failed',type(exc).__name__))
                    else:
                        try:
                            conn.commit()
                            result.extend(outcomes([row],'inserted'))
                        except Exception as exc:
                            raise RuntimeError('Commit outcome unknown; reconcile by TRIP_ID before retry') from exc
                return result
            else:
                try:
                    conn.commit()
                except Exception as exc:
                    raise RuntimeError('Batch commit outcome unknown; reconcile by TRIP_ID before retry') from exc
                return outcomes(rows,'inserted')
        for row in iterator:
            pending.append(row.asDict())
            if len(pending)==BATCH_SIZE:
                result=flush(pending); pending=[]
                yield from result
        if pending:
            result=flush(pending); pending=[]
            yield from result
    except Exception as exc:
        if conn is not None:
            try: conn.rollback()
            except Exception: pass
        yield from outcomes(pending,'failed_or_unknown',type(exc).__name__)
        for row in iterator:
            yield from outcomes([row.asDict()],'failed_or_unknown','PartitionConnectionUnavailable')
    finally:
        if conn is not None:
            try: conn.close()
            except Exception: pass


**Expected check:** The function is defined but not executed. Fallback handles the full original row plus outcome/reason.


## Step 4 — Execute once and persist the returned outcomes

Validate database identifiers, do not print queries or credentials, and perform a single bounded collect of write outcomes. Spark task retries/speculation still make external side effects at least once; unique keys and reconciliation are mandatory.


In [ ]:
if RUN_DB2:
    assert all([DB2_URL,DB2_JAR,DB2_TABLE,SECRET_SCOPE,USER_KEY,PASSWORD_KEY])
    assert re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*',DB2_TABLE)
    db2_user=dbutils.secrets.get(SECRET_SCOPE,USER_KEY)
    db2_password=dbutils.secrets.get(SECRET_SCOPE,PASSWORD_KEY)
    insert_sql=f'INSERT INTO {DB2_TABLE} (TRIP_ID, PICKUP_ZIP, FARE, PICKUP_TS) VALUES (?, ?, ?, ?)'
    exists_sql=f'SELECT COUNT(*) FROM {DB2_TABLE} WHERE TRIP_ID = ?'
    # Executor credentials are serialized for this explicit integration. Restrict job/cluster access,
    # use a minimally privileged service account, and follow your organization’s secret policy.
    outcomes=export.repartition(NUM_PARTITIONS).rdd.mapPartitions(partition_writer).collect()
    outcome_schema='trip_id string, pickup_zip string, fare decimal(14,2), pickup_ts timestamp, status string, reason string'
    results=spark.createDataFrame(outcomes,outcome_schema)
    assert results.count()==export.count(), 'Incomplete outcomes: reconcile target before rerunning.'
    results.write.format('delta').mode('overwrite').saveAsTable(table('db2_outcomes'))
    results.filter("status LIKE 'failed%'").write.format('delta').mode('overwrite').saveAsTable(table('db2_rejects'))
    counts={r['status']:r['count'] for r in results.groupBy('status').count().collect()}
    summary={'total':len(outcomes),'inserted':counts.get('inserted',0),'skipped':counts.get('skipped',0),
        'failed':counts.get('failed',0),'failed_or_unknown':counts.get('failed_or_unknown',0)}
    assert summary['total']==sum(summary[k] for k in ['inserted','skipped','failed','failed_or_unknown'])
    print(summary)
else:
    print('Optional DB2 execution skipped. No queries or credentials printed.')


**Expected check:** Enabled mode retains full exported rows plus status/reason and prints only summary counts.


## Practice

Add an idempotency ledger keyed by source business key plus source version. Test a lost connection around commit in a dedicated test database.

## Solution guide

An inserted count is local to the observed execution; a retry may report previously inserted rows as skipped. Reconcile target keys and payloads for durable totals. Do not claim exactly-once effects from Spark execution alone.

## Production considerations

Keep source lineage and stable business keys. Measure on representative volumes, review nulls and time semantics, and make retries idempotent. Sample-data checks demonstrate mechanics; they do not establish production performance or exactly-once external writes.
